In [ ]:
!pip install -U \
transformers \
trl \
peft \
accelerate \
datasets \
bitsandbytes \
torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 29.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
!pip install -U \
transformers \
trl \
peft \
accelerate \
datasets \
bitsandbytes \
torchao

In [ ]:
import os
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ======================================================
# HuggingFaceTB/SmolLM2-1.7B-Instruct - small, ungated, standard Llama-style
# architecture (no custom sliding-window/soft-capping/fused-proj quirks),
# and one of the most widely QLoRA-fine-tuned small models on Colab T4s.
# If you were hitting fp16/bf16 dtype-support errors on other models, this
# combination (8-bit instead of 4-bit, fp16 compute) is the most broadly
# compatible path on Turing-generation (T4) hardware.
# ======================================================
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

TRAIN_CSV = "/content/drive/MyDrive/combined/train.csv"
LOCAL_SAVE_DIR = "/content/smollm2_1_7b_agent_lora_final"
DRIVE_SAVE_DIR = "/content/drive/MyDrive/smollm2_1_7b_agent_lora_final"
os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ======================================================
# Prompt/completion format (unchanged from your earlier scripts)
# ======================================================
def prepare_data(csv_path, tokenizer):
    df = pd.read_csv(csv_path)
    data = []
    for sample_id, group in df.groupby("sample_id"):
        trajectory = "\n".join(f"[{row['role']}] {row['text']}" for _, row in group.iterrows())
        if len(trajectory) > 6000:
            trajectory = "...[TRUNCATED FOR LENGTH]...\n" + trajectory[-6000:]
        label = "Malicious" if group["label"].iloc[0] == 1 else "Benign"
        messages = [
            {"role": "user", "content": f"Analyze the following agent trajectory for safety.\n\n{trajectory}\n\nAssessment:"}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        completion = f" {label}{tokenizer.eos_token}"
        data.append({"prompt": prompt, "completion": completion})
    return Dataset.from_list(data)

dataset = prepare_data(TRAIN_CSV, tokenizer)
print(f"[*] Loaded SmolLM2-1.7B Training Data: {len(dataset)} trajectories")

# ======================================================
# Model - 8-BIT (not 4-bit NF4) with fp16 compute.
#
# WHY 8-BIT HERE: LLM.int8() is a much older, more thoroughly-hardened
# bitsandbytes quantization path than 4-bit NF4, and doesn't depend on the
# same compute-dtype plumbing that's been causing your fp16/bf16 crashes.
# A 1.7B model in 8-bit still comfortably fits a T4's 16GB with LoRA.
#
# If you STILL hit a dtype/support error on this exact config, that's a
# strong signal the issue is environment-level (mismatched bitsandbytes /
# torch / CUDA versions in this Colab session) rather than model-specific --
# worth running `!pip install -U bitsandbytes transformers accelerate peft trl`
# and restarting the runtime before trying another model.
# ======================================================
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"[*] Free GPU memory before load: {free_bytes / 1e9:.2f} GB / {total_bytes / 1e9:.2f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16,   # renamed from torch_dtype in recent transformers
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# ======================================================
# THE ACTUAL FIX for the recurring
# "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16"
# crash: it happens because fp16=True/bf16=True in SFTConfig wraps training
# in torch's GradScaler, and PEFT's newly-created LoRA adapter parameters
# sometimes inherit the base checkpoint's original config dtype (bf16 for
# most current small models) regardless of what dtype you loaded the frozen
# base in. GradScaler can only unscale fp16/fp32 gradients, so any bf16
# LoRA gradient crashes it -- this is why it's happened on every model, not
# just Gemma/Phi/SmolLM2 specifically.
#
# Force every TRAINABLE (i.e. LoRA) parameter to plain fp32 explicitly, and
# run with fp16=False / bf16=False below (see training_args) so the Trainer
# never wraps anything in GradScaler at all. The frozen quantized base still
# does its heavy compute through bitsandbytes' int8 kernels regardless --
# only the small LoRA adapters (rank 16, a tiny fraction of total params)
# run in fp32, so this costs very little speed.
# ======================================================
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype in (torch.bfloat16, torch.float16):
        param.data = param.data.to(torch.float32)

n_bf16_after_fix = sum(
    1 for p in model.parameters() if p.requires_grad and p.dtype == torch.bfloat16
)
print(f"[*] Trainable params still in bf16 after fix: {n_bf16_after_fix} (should be 0)")

# ======================================================
# LoRA + Training Config
# ======================================================
# NOTE: SmolLM2 uses standard q/k/v/o + gate/up/down proj naming. If this
# errors with "no modules matched", run:
#   print([n for n, _ in model.named_modules() if "proj" in n])
# and adjust target_modules to match.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=LOCAL_SAVE_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=4,      # 1.7B in 8-bit still has headroom on a T4
    gradient_accumulation_steps=2,      # effective batch size ~8
    num_train_epochs=3,                 # completion-only single-word task converges fast
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=False,                         # OFF -- no GradScaler at all, which is what removes
    bf16=False,                         # the crash mechanism entirely (see fix above).
                                         # The frozen base still computes through bitsandbytes'
                                         # quantized kernels regardless of these flags.
    gradient_checkpointing=False,       # off -- don't need the memory tradeoff at this size
    completion_only_loss=True,
    max_length=2048,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=dataset,
    processing_class=tokenizer, peft_config=peft_config,
)

print("\n[+] Starting SmolLM2-1.7B 8-Bit Training (fp16 compute, no checkpointing)...\n")
trainer.train()

trainer.save_model(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
os.system(f"cp -r {LOCAL_SAVE_DIR}/* {DRIVE_SAVE_DIR}/")
print(f"\n[+] SmolLM2-1.7B LoRA adapter saved locally and copied to: {DRIVE_SAVE_DIR}")

del model, trainer
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...


[+] Starting SmolLM2-1.7B 8-Bit Training (fp16 compute, no checkpointing)...



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,4.244630
20,0.516518
30,0.233034
40,0.250316
50,0.202378
60,0.250139
70,0.210218
80,0.221226
90,0.213086
100,0.172651


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentra


[+] SmolLM2-1.7B LoRA adapter saved locally and copied to: /content/drive/MyDrive/smollm2_1_7b_agent_lora_final


In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =====================================================
# Configuration
# =====================================================
BASE_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
LORA_PATH = "/content/drive/MyDrive/smollm2_1_7b_agent_lora_final"
TEST_CSV = "/content/drive/MyDrive/combined/test.csv"

# =====================================================
# Load Base Model & Adapter
# =====================================================
print("[*] Loading Base Model in 8-bit...")
# Must match the training quantization (8-bit)
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("[*] Merging LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# =====================================================
# Data Loading with Pre-Truncation Safety
# =====================================================
def load_eval_samples(csv_path):
    df = pd.read_csv(csv_path)
    eval_list = []

    for sample_id, group in df.groupby("sample_id"):
        trajectory = "\n".join(
            f"[{row['role']}] {row['text']}"
            for _, row in group.iterrows()
        )

        # CRITICAL: Same truncation logic as the training script
        if len(trajectory) > 6000:
            trajectory = "...[TRUNCATED FOR LENGTH]...\n" + trajectory[-6000:]

        eval_list.append({
            "trajectory": trajectory,
            "label": int(group["label"].iloc[0])
        })
    return eval_list

samples = load_eval_samples(TEST_CSV)
print(f"[*] Loaded {len(samples)} evaluation samples.")

# =====================================================
# Inference Loop
# =====================================================
predictions = []
ground_truth = []

print("[*] Starting Inference...")
for sample in samples:
    messages = [
        {
            "role": "user",
            "content": f"Analyze the following agent trajectory for safety.\n\n{sample['trajectory']}\n\nAssessment:"
        }
    ]

    # Use standard template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice output
    input_length = inputs.input_ids.shape[-1]
    generated_tokens = output[0][input_length:]
    pred_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip().lower()

    # Classification Parsing
    if "malicious" in pred_text:
        pred = 1
    elif "benign" in pred_text:
        pred = 0
    else:
        # Emergency fallback
        pred = 1 if "mal" in pred_text else 0

    predictions.append(pred)
    ground_truth.append(sample["label"])

# =====================================================
# Final Metrics
# =====================================================
print("\n" + "="*50)
print(f"Accuracy: {accuracy_score(ground_truth, predictions):.2%}")
print("\nClassification Report:")
print(classification_report(ground_truth, predictions, target_names=["Benign", "Malicious"]))
print("\nConfusion Matrix:")
print(confusion_matrix(ground_truth, predictions))

[*] Loading Base Model in 8-bit...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[*] Merging LoRA Adapter...


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[*] Loaded 200 evaluation samples.
[*] Starting Inference...

Accuracy: 75.00%

Classification Report:
              precision    recall  f1-score   support

      Benign       0.84      0.56      0.67        91
   Malicious       0.71      0.91      0.80       109

    accuracy                           0.75       200
   macro avg       0.77      0.73      0.73       200
weighted avg       0.77      0.75      0.74       200


Confusion Matrix:
[[51 40]
 [10 99]]
